# PR Governance Agent - Live Demo

**Fully self-contained.** Just set your GitHub token and repo below, then click **Runtime > Run all**.
The notebook will automatically create a branch, open a PR, trigger the agent, and show results — all inline.

> **You need**: A GitHub PAT with `repo` scope. Get one at https://github.com/settings/tokens

## Step 1: Configure

In [ ]:
from getpass import getpass
import urllib.request, json, time, re

# Your GitHub repo to demo against
REPO = "lmudu2/risk-analyzer-poc"

# GitHub PAT with repo scope (input is hidden)
GITHUB_TOKEN = getpass("GitHub PAT (repo scope required): ")

GATEWAY_URL = "https://zrfcfcrgrwublxw22es6gvuidq0ckmpm.lambda-url.us-east-1.on.aws/"

GH_HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json",
    "Content-Type": "application/json",
    "User-Agent": "PR-Agent-Demo"
}

def gh_get(path):
    url = f"https://api.github.com/repos/{REPO}/{path}"
    req = urllib.request.Request(url, headers=GH_HEADERS)
    with urllib.request.urlopen(req, timeout=10) as r:
        return json.loads(r.read())

def gh_post(path, data):
    url = f"https://api.github.com/repos/{REPO}/{path}"
    req = urllib.request.Request(url, data=json.dumps(data).encode(), headers=GH_HEADERS, method='POST')
    with urllib.request.urlopen(req, timeout=15) as r:
        return json.loads(r.read())

print(f"Target:  {REPO}")
print(f"Agent:   {GATEWAY_URL}")
print("Ready!")

## Step 2: Auto-Create a Test PR

In [ ]:
# Get the default branch SHA to branch off
repo_info = gh_get("")
default_branch = repo_info["default_branch"]
branch_info = gh_get(f"git/refs/heads/{default_branch}")
base_sha = branch_info["object"]["sha"]

# Create a new branch
branch_name = f"demo-agent-test-{int(time.time())}"
gh_post("git/refs", {"ref": f"refs/heads/{branch_name}", "sha": base_sha})
print(f"Branch created: {branch_name}")

# Add a test file to the branch
import base64
file_content = base64.b64encode(f"# Demo test file\nCreated at {time.strftime('%Y-%m-%d %H:%M:%S')}\n".encode()).decode()
gh_post(f"contents/demo_test_{int(time.time())}.md", {
    "message": f"demo: add test file for agent demo",
    "content": file_content,
    "branch": branch_name
})
print("Test file committed to branch")

# Open the PR
pr = gh_post("pulls", {
    "title": f"[DEMO] Agent test - {time.strftime('%H:%M:%S')}",
    "head": branch_name,
    "base": default_branch,
    "body": "This PR was auto-created by the PR Governance Agent demo notebook."
})
PR_NUMBER = str(pr["number"])
PR_URL = pr["html_url"]
print(f"\nPR #{PR_NUMBER} created: {PR_URL}")
print(f"Watching for the agent to respond...")

## Step 3: Trigger the Agent (HIGH RISK Simulation)

In [ ]:
# Send a HIGH RISK payload pointing at the real PR we just created
high_risk = {
    "action": "opened",
    "pull_request": {
        "number": int(PR_NUMBER),
        "title": "Refactor: drop user_id column from users table",
        "user": {"login": "demo-presenter", "type": "User"}
    },
    "repository": {"full_name": REPO},
    "installation": {"id": 12345678},
    "sender": {"login": "demo-presenter", "type": "User"}
}
req = urllib.request.Request(GATEWAY_URL, data=json.dumps(high_risk).encode(),
    headers={'Content-Type': 'application/json', 'X-GitHub-Event': 'pull_request'}, method='POST')
with urllib.request.urlopen(req, timeout=30) as res:
    print(f"HIGH RISK payload sent to agent (HTTP {res.getcode()})")
print(f"Watch live: {PR_URL}")
print("Agent will: HIGH RISK -> auto-close PR -> create Jira ticket -> send approval email")

## Step 4: View Live Results

In [ ]:
print("Waiting 20 seconds for agent to finish...")
time.sleep(20)

# PR Status
pr_data = gh_get(f"pulls/{PR_NUMBER}")
state = pr_data.get('state','?').upper()
merged = pr_data.get('merged', False)
icon = 'MERGED' if merged else ('CLOSED - blocked by agent!' if state=='CLOSED' else 'OPEN')
print(f"\n=== PR STATUS ===")
print(f"PR #{PR_NUMBER}: {icon}")
print(f"URL: {PR_URL}")

# Agent comment
comments = gh_get(f"issues/{PR_NUMBER}/comments")
jira = None
print(f"\n=== AI RISK REPORT ===")
for c in reversed(comments):
    body = c.get('body','')
    if 'RISK' in body.upper():
        print(body[:1000])
        m = re.search(r'([A-Z]+-\d+)', body)
        if m: jira = m.group(1)
        break
else:
    print("Agent comment not posted yet - re-run this cell in ~10 seconds")

# Jira
print(f"\n=== JIRA TICKET ===")
if jira:
    print(f"Created: {jira}")
    print(f"View: https://lmudu95.atlassian.net/browse/{jira}")
else:
    print("No ticket found yet (only created for HIGH RISK PRs)")

# Email
print(f"\n=== EMAIL ===")
if state == 'CLOSED' and not merged:
    print("Approval email sent to lmudu95@gmail.com (PR was auto-closed = HIGH RISK)")
elif merged:
    print("No email - PR was auto-merged (LOW RISK)")
else:
    print("Still processing - re-run this cell")